# Sesi 5 - Knowledge ERP: CRUD REST API

Bagian dari kurikulum **Agentic Knowledge System - FastAPI + DuckDB + Qwen (Local, tanpa LLM cloud)**.

> Jalankan sel dari atas ke bawah secara berurutan. Sel bertanda `🔨 TODO` wajib dilengkapi peserta; solusi contoh disediakan pada sel setelahnya (bisa disembunyikan saat sesi diberikan ke peserta).

## 0. Setup Environment

In [ ]:
!pip -q install duckdb fastapi httpx

## 1. Skema Data ERP
Domain kedua: data transaksional (produk, pelanggan, order) - beda karakter dari dokumen bebas di Sesi 1-4, karena di sini agent nanti akan **menulis** data, bukan cuma membaca.

In [ ]:
import duckdb, uuid

erp_con = duckdb.connect("erp.duckdb")
erp_con.execute("""
CREATE TABLE IF NOT EXISTS products (
    id VARCHAR PRIMARY KEY, name VARCHAR, price DOUBLE, stock INTEGER
);
CREATE TABLE IF NOT EXISTS customers (
    id VARCHAR PRIMARY KEY, name VARCHAR, email VARCHAR
);
CREATE TABLE IF NOT EXISTS orders (
    id VARCHAR PRIMARY KEY, customer_id VARCHAR, status VARCHAR,
    created_at TIMESTAMP DEFAULT current_timestamp
);
CREATE TABLE IF NOT EXISTS order_items (
    id VARCHAR PRIMARY KEY, order_id VARCHAR, product_id VARCHAR,
    qty INTEGER, price DOUBLE
);
""")
print("Skema ERP siap.")

## 2. Seed Data Contoh

In [ ]:
if erp_con.execute("SELECT COUNT(*) FROM products").fetchone()[0] == 0:
    products_seed = [
        ("Kaos Polos Hitam", 75000, 50),
        ("Kaos Polos Putih", 75000, 40),
        ("Jaket Hoodie", 250000, 15),
    ]
    for name, price, stock in products_seed:
        erp_con.execute("INSERT INTO products VALUES (?, ?, ?, ?)",
                        [str(uuid.uuid4()), name, price, stock])

if erp_con.execute("SELECT COUNT(*) FROM customers").fetchone()[0] == 0:
    erp_con.execute("INSERT INTO customers VALUES (?, ?, ?)",
                    [str(uuid.uuid4()), "Budi Santoso", "budi@example.com"])

erp_con.execute("SELECT * FROM products").fetchdf()

## 3. Bangun FastAPI App - CRUD Produk, Pelanggan, Order

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List

erp_app = FastAPI(title="Knowledge ERP API")

class ProductIn(BaseModel):
    name: str
    price: float
    stock: int

class ProductOut(ProductIn):
    id: str

@erp_app.post("/products", response_model=ProductOut)
def create_product(p: ProductIn):
    pid = str(uuid.uuid4())
    erp_con.execute("INSERT INTO products VALUES (?, ?, ?, ?)", [pid, p.name, p.price, p.stock])
    return {"id": pid, **p.dict()}

@erp_app.get("/products", response_model=List[ProductOut])
def list_products():
    rows = erp_con.execute("SELECT id, name, price, stock FROM products").fetchall()
    return [{"id": r[0], "name": r[1], "price": r[2], "stock": r[3]} for r in rows]

@erp_app.get("/products/{pid}", response_model=ProductOut)
def get_product(pid: str):
    row = erp_con.execute("SELECT id, name, price, stock FROM products WHERE id=?", [pid]).fetchone()
    if not row:
        raise HTTPException(404, "Product not found")
    return {"id": row[0], "name": row[1], "price": row[2], "stock": row[3]}

### 🔨 TODO 1
Lengkapi `PUT /products/{pid}` dan `DELETE /products/{pid}`, mengikuti pola CRUD dari Sesi 2.

In [ ]:
# TODO: implementasikan update_product & delete_product
@erp_app.put("/products/{pid}", response_model=ProductOut)
def update_product(pid: str, p: ProductIn):
    pass

@erp_app.delete("/products/{pid}")
def delete_product(pid: str):
    pass

**Solusi contoh:**
```python
@erp_app.put("/products/{pid}", response_model=ProductOut)
def update_product(pid: str, p: ProductIn):
    erp_con.execute("UPDATE products SET name=?, price=?, stock=? WHERE id=?",
                    [p.name, p.price, p.stock, pid])
    return {"id": pid, **p.dict()}

@erp_app.delete("/products/{pid}")
def delete_product(pid: str):
    erp_con.execute("DELETE FROM products WHERE id=?", [pid])
    return {"status": "deleted", "id": pid}
```

## 4. Endpoint Order (dengan validasi stok)

In [ ]:
class OrderItemIn(BaseModel):
    product_id: str
    qty: int

class OrderIn(BaseModel):
    customer_id: str
    items: List[OrderItemIn]

@erp_app.post("/orders")
def create_order(o: OrderIn):
    for item in o.items:
        stock = erp_con.execute("SELECT stock FROM products WHERE id=?", [item.product_id]).fetchone()
        if not stock:
            raise HTTPException(404, f"Produk {item.product_id} tidak ditemukan")
        if stock[0] < item.qty:
            raise HTTPException(400, f"Stok tidak cukup untuk produk {item.product_id}")

    order_id = str(uuid.uuid4())
    erp_con.execute("INSERT INTO orders VALUES (?, ?, 'pending', current_timestamp)",
                    [order_id, o.customer_id])
    for item in o.items:
        price = erp_con.execute("SELECT price FROM products WHERE id=?", [item.product_id]).fetchone()[0]
        erp_con.execute("INSERT INTO order_items VALUES (?, ?, ?, ?, ?)",
                        [str(uuid.uuid4()), order_id, item.product_id, item.qty, price])
        erp_con.execute("UPDATE products SET stock = stock - ? WHERE id=?", [item.qty, item.product_id])
    return {"order_id": order_id, "status": "pending"}

@erp_app.get("/orders/{order_id}")
def get_order(order_id: str):
    order = erp_con.execute("SELECT id, customer_id, status FROM orders WHERE id=?", [order_id]).fetchone()
    if not order:
        raise HTTPException(404, "Order not found")
    items = erp_con.execute("""
        SELECT p.name, oi.qty, oi.price FROM order_items oi
        JOIN products p ON p.id = oi.product_id WHERE oi.order_id = ?
    """, [order_id]).fetchall()
    return {
        "id": order[0], "customer_id": order[1], "status": order[2],
        "items": [{"product": i[0], "qty": i[1], "price": i[2]} for i in items]
    }

## 5. Uji API dengan TestClient

In [ ]:
from fastapi.testclient import TestClient

erp_client = TestClient(erp_app)

products = erp_client.get("/products").json()
customer = erp_con.execute("SELECT id FROM customers LIMIT 1").fetchone()[0]

order_resp = erp_client.post("/orders", json={
    "customer_id": customer,
    "items": [{"product_id": products[0]["id"], "qty": 2}]
})
print(order_resp.status_code, order_resp.json())

In [ ]:
erp_client.get(f"/orders/{order_resp.json()['order_id']}").json()

### 🔨 TODO 2
Uji kasus order dengan `qty` melebihi stok yang tersedia. Pastikan respons berupa error 400 yang informatif, bukan crash.

In [ ]:
# TODO: uji order dengan qty berlebih
bad_order = erp_client.post("/orders", json={
    "customer_id": customer,
    "items": [{"product_id": products[0]["id"], "qty": 9999}]
})
print(bad_order.status_code, bad_order.json())

## Ringkasan Sesi 5
API ERP (`erp_app`) sudah punya CRUD produk + order dengan validasi stok. Sesi 6 akan menghubungkan API ini ke ReAct agent yang bisa mengambil **aksi**, bukan hanya menjawab.